# Instruction the lab-work #1
`TODO`: make the final .ipynb file in format `"{second_name}_{first_name}_{group_id}__lab_1.ipynb"`

First name     - Volodymyr

Second name    - Donets

Group          - KU-41

# Task description

## Overall description

In this lab work, you will get familiar with specific computer vision models for solving object detection (predicting Bounding Boxes / Regions of Interest), image segmentation (pixel-level classification), and gain practical experience with object tracking models across video frames. 

Additionally, you will train your own object detection model (from the YOLO family, MobileNet family, or similar) on a custom dataset using a standardized library.


## *Instructions for the work*

1. **Dataset Selection & Preparation**: Choose an image dataset that interests you (e.g., in YOLO or COCO format). Perform the necessary data preprocessing, augmentations, and split the data into training, validation, and test sets.
2. **Review Baselines**: Carefully study the provided examples to understand the standard computer vision pipeline, from loading images/videos to rendering predictions.
3. **Pre-trained Inference & Tracking**: Apply pre-trained models to perform zero-shot or out-of-the-box object detection, segmentation, and tracking on a sample video or set of images (use your own or look at prepared examples of data).
4. **Custom Model Training**: Fine-tune a lightweight object detection model (e.g., YOLOv10-nano, SSD MobileNet) on your selected dataset. Monitor training metrics like loss and mean Average Precision (mAP), Intersection over Union (IoU).
5. **Evaluation**: Analyze the results and write a conclusion. Compare the performance (mAP, IoU) and inference speed (FPS) of your custom-trained model against the pre-trained baseline.
6. **BONUS TASK**: Run one of the presented models on your phone (yolov10-nano, SSD MobileNet v3). It should be app like a camera-app, but with object detection showing bounding boxes and class labels. // OR // Implement real-time inference or deployment. Run your trained model in real-time using your computer's webcam, or wrap the model in a simple local web interface (using Gradio or Streamlit) where a user can upload an image and get predictions.
## Examples

1. 

## Important

1. **No Single-Cell Solutions**: Do NOT write all your code in a single Jupyter Notebook cell. Your code must be logically separated. Single-cell notebooks will receive a grade of 0. If you need help structuring a notebook, ask the instructor or your peers. Alternatively, you may submit well-structured .py files alongside a .pdf report of your results.
2. **Academic Integrity**: Plagiarism is prohibited. Identical or copied works will automatically receive a grade of 0.
3. **Submission Format**: You are encouraged to solve this assignment in a separate .ipynb file. You may duplicate the provided template, clear the existing solution, and build your own.
4. **Mandatory Conclusions**: A project without a written analytical conclusion is incomplete. Do not forget to summarize your findings at the end of your notebook.

## How to get max num of points without personal work presentation.

As I told you, you can get the maximum mark for all lab works without personal presentation in case of using something different and additional depth of work.
So for this work you can:

1. `Track A`: Compare the performance of tiny and large object detection models, or compare fundamentally different architectures (e.g., YOLO vs. Faster R-CNN) trained on your data.
2. `Track B`: Gather and annotate your own custom data (non-Kaggle). For example, creating a dataset to detect your own socks, your pets, or specific items on your desk.

# 1. Import dependencies

In [ ]:
# Install required dependencies
%pip install opencv-python matplotlib numpy pillow torch torchvision mediapipe

In [ ]:
import os
import glob
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Set plotting style for clear visualizations
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['image.interpolation'] = 'nearest'

# 2. Tool functions

In [ ]:
def load_image(image_path: str) -> np.ndarray:
    """
    Load an image from disk and convert BGR (OpenCV default) to RGB.
    """
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(f"Image not found at path: {image_path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return img_rgb


def draw_bounding_boxes(
    image: np.ndarray,
    boxes: list,
    color: tuple = (255, 0, 0),
    thickness: int = 2,
    label: str = None
) -> np.ndarray:
    """
    Draw bounding boxes on a copy of the RGB image.
    Each box is in format (x, y, w, h).
    """
    img_copy = image.copy()
    for (x, y, w, h) in boxes:
        cv2.rectangle(img_copy, (x, y), (x + w, y + h), color, thickness)
        if label:
            label_size, _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)
            cv2.rectangle(
                img_copy,
                (x, max(0, y - label_size[1] - 8)),
                (x + label_size[0] + 4, max(0, y)),
                color,
                -1
            )
            cv2.putText(
                img_copy,
                label,
                (x + 2, max(label_size[1] + 2, y - 4)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                1,
                cv2.LINE_AA
            )
    return img_copy


def visualize_detections(
    original_img: np.ndarray,
    detected_img: np.ndarray,
    title: str = "Detection Result"
):
    """
    Display original and detected images side-by-side.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(original_img)
    axes[0].set_title("Original Image", fontsize=14)
    axes[0].axis("off")

    axes[1].imshow(detected_img)
    axes[1].set_title(title, fontsize=14)
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Configure source folder for human ROI test images
DATA_DIR = os.path.join("src", "human_ROI_test")
if not os.path.exists(DATA_DIR):
    DATA_DIR = os.path.join("lab_works_2026_2027_1st_sem", "lab_1", "src", "human_ROI_test")

image_extensions = ("*.jpg", "*.jpeg", "*.png")

# Collect all available test images in source folder
image_paths = []
for ext in image_extensions:
    image_paths.extend(glob.glob(os.path.join(DATA_DIR, ext)))
image_paths = sorted(image_paths)

print(f"Found {len(image_paths)} images in '{DATA_DIR}':")
for p in image_paths:
    print(f" - {os.path.basename(p)}")

# 3. Running object detection models (ROI detection)

## 3.1. HaaR Cascades -- object detection method from classical ML era

In [ ]:
# Load pre-trained Haar cascade models from OpenCV data directory
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_eye.xml")
fullbody_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_fullbody.xml")
upperbody_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_upperbody.xml")

print("Haar Cascade Classifiers loaded successfully:")
print(" - Frontal Face:", not face_cascade.empty())
print(" - Eyes:", not eye_cascade.empty())
print(" - Full Body:", not fullbody_cascade.empty())
print(" - Upper Body:", not upperbody_cascade.empty())


def detect_haar_objects(
    image: np.ndarray,
    detect_faces: bool = True,
    detect_eyes: bool = True,
    detect_fullbody: bool = False,
    detect_upperbody: bool = False,
    scale_factor: float = 1.1,
    min_neighbors: int = 5,
    min_size: tuple = (30, 30)
) -> tuple[np.ndarray, dict]:
    """
    Perform Haar Cascade detection on RGB image and return annotated image and detection coordinates.
    """
    result_img = image.copy()
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    detections = {}

    if detect_fullbody:
        bodies = fullbody_cascade.detectMultiScale(
            gray, scaleFactor=1.05, minNeighbors=3, minSize=(50, 100)
        )
        detections['fullbody'] = bodies
        result_img = draw_bounding_boxes(result_img, bodies, color=(0, 255, 0), thickness=2, label="Body")

    if detect_upperbody:
        upper_bodies = upperbody_cascade.detectMultiScale(
            gray, scaleFactor=1.05, minNeighbors=3, minSize=(50, 50)
        )
        detections['upperbody'] = upper_bodies
        result_img = draw_bounding_boxes(result_img, upper_bodies, color=(255, 165, 0), thickness=2, label="UpperBody")

    if detect_faces:
        faces = face_cascade.detectMultiScale(
            gray, scaleFactor=scale_factor, minNeighbors=min_neighbors, minSize=min_size
        )
        detections['faces'] = faces
        result_img = draw_bounding_boxes(result_img, faces, color=(255, 0, 0), thickness=2, label="Face")

        if detect_eyes:
            for (fx, fy, fw, fh) in faces:
                roi_gray = gray[fy:fy+fh, fx:fx+fw]
                roi_color = result_img[fy:fy+fh, fx:fx+fw]
                eyes = eye_cascade.detectMultiScale(roi_gray, scaleFactor=1.1, minNeighbors=4, minSize=(15, 15))
                for (ex, ey, ew, eh) in eyes:
                    cv2.rectangle(roi_color, (ex, ey), (ex + ew, ey + eh), (0, 255, 255), 2)
                    cv2.putText(roi_color, "Eye", (ex, max(ey - 2, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)

    return result_img, detections

In [ ]:
# Visualize Haar Cascade detection results across test images from human_ROI_test
for img_path in image_paths:
    img_name = os.path.basename(img_path)
    img = load_image(img_path)

    # Run detection: faces, eyes, and full body / upper body
    detected_img, detections = detect_haar_objects(
        img,
        detect_faces=True,
        detect_eyes=True,
        detect_fullbody=True,
        detect_upperbody=True,
        scale_factor=1.1,
        min_neighbors=5,
        min_size=(30, 30)
    )

    num_faces = len(detections.get('faces', []))
    num_bodies = len(detections.get('fullbody', []))
    num_upper = len(detections.get('upperbody', []))

    title = f"{img_name} | Faces: {num_faces}, Bodies: {num_bodies}, UpperBodies: {num_upper}"
    visualize_detections(img, detected_img, title=title)

## 3.2. YOLOv10 nano (trained on COCO dataset)

* [HugginFace model source](https://huggingface.co/jameslahm/yolov10n) -- if you want to deal with Ultralitics
* [Git Repo of the Devs](https://github.com/THU-MIG/yolov10)
* [Article](https://arxiv.org/abs/2405.14458)
* [Git model source](https://github.com/THU-MIG/yolov10/releases) -- ONNX model to avoid Ultralitics

In [ ]:
# COCO 80 class labels
COCO_CLASSES = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light',
    'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee',
    'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard',
    'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple',
    'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch',
    'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
    'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear',
    'hair drier', 'toothbrush'
]

# Generate unique deterministic colors for each COCO class for clear visualization
np.random.seed(42)
CLASS_COLORS = {
    cls_name: tuple(int(c) for c in np.random.randint(50, 255, size=3))
    for cls_name in COCO_CLASSES
}

# Locate and load YOLOv10 nano ONNX model
YOLO_MODEL_PATH = os.path.join("models", "yolov10n.onnx")
if not os.path.exists(YOLO_MODEL_PATH):
    YOLO_MODEL_PATH = os.path.join("lab_works_2026_2027_1st_sem", "lab_1", "models", "yolov10n.onnx")

yolo_net = cv2.dnn.readNetFromONNX(YOLO_MODEL_PATH)
print(f"YOLOv10 nano model loaded successfully from: {YOLO_MODEL_PATH}")
print(f"Total COCO classes supported: {len(COCO_CLASSES)}")

In [ ]:
def letterbox_preprocess(
    image: np.ndarray,
    target_shape: tuple = (640, 640),
    pad_color: tuple = (114, 114, 114)
) -> tuple[np.ndarray, float, tuple[float, float]]:
    """
    Resize and pad image while preserving aspect ratio.
    Returns: padded_image, scale_ratio, (pad_width, pad_height)
    """
    h, w = image.shape[:2]
    target_h, target_w = target_shape
    scale = min(target_w / w, target_h / h)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))

    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    pad_w = (target_w - new_w) / 2
    pad_h = (target_h - new_h) / 2
    top, bottom = int(round(pad_h - 0.1)), int(round(pad_h + 0.1))
    left, right = int(round(pad_w - 0.1)), int(round(pad_w + 0.1))

    padded = cv2.copyMakeBorder(
        resized, top, bottom, left, right,
        cv2.BORDER_CONSTANT, value=pad_color
    )
    return padded, scale, (pad_w, pad_h)


def detect_yolov10_objects(
    image: np.ndarray,
    net: cv2.dnn.Net = yolo_net,
    conf_threshold: float = 0.30,
    input_size: tuple = (640, 640)
) -> tuple[np.ndarray, list[dict]]:
    """
    Perform YOLOv10 object detection on an RGB image.
    Returns annotated RGB image and list of detection dictionaries.
    """
    h, w = image.shape[:2]
    padded_img, scale, (pad_w, pad_h) = letterbox_preprocess(image, target_shape=input_size)

    # YOLO blob preprocessing: normalize 1/255.0, RGB format
    blob = cv2.dnn.blobFromImage(padded_img, scalefactor=1.0 / 255.0, size=input_size, swapRB=False, crop=False)
    net.setInput(blob)
    predictions = net.forward()[0]  # Shape: (300, 6) -> [x1, y1, x2, y2, score, class_id]

    result_img = image.copy()
    detections = []

    for det in predictions:
        x1, y1, x2, y2, score, cls_id = det
        if score >= conf_threshold:
            cls_id = int(cls_id)
            cls_name = COCO_CLASSES[cls_id] if 0 <= cls_id < len(COCO_CLASSES) else f"class_{cls_id}"

            # Map coordinates back from padded letterbox space to original image coordinates
            orig_x1 = max(0, min(w, int(round((x1 - pad_w) / scale))))
            orig_y1 = max(0, min(h, int(round((y1 - pad_h) / scale))))
            orig_x2 = max(0, min(w, int(round((x2 - pad_w) / scale))))
            orig_y2 = max(0, min(h, int(round((y2 - pad_h) / scale))))

            box_w = orig_x2 - orig_x1
            box_h = orig_y2 - orig_y1

            color = CLASS_COLORS.get(cls_name, (0, 255, 0))
            label = f"{cls_name} {score:.2f}"

            # Draw bounding box and label badge
            cv2.rectangle(result_img, (orig_x1, orig_y1), (orig_x2, orig_y2), color, 2)
            label_size, _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            badge_y1 = max(0, orig_y1 - label_size[1] - 6)
            badge_y2 = max(0, orig_y1)
            cv2.rectangle(
                result_img,
                (orig_x1, badge_y1),
                (orig_x1 + label_size[0] + 6, badge_y2),
                color,
                -1
            )
            cv2.putText(
                result_img,
                label,
                (orig_x1 + 3, max(label_size[1] + 1, orig_y1 - 3)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (255, 255, 255),
                1,
                cv2.LINE_AA
            )

            detections.append({
                "class_id": cls_id,
                "class_name": cls_name,
                "score": float(score),
                "box": (orig_x1, orig_y1, box_w, box_h)
            })

    return result_img, detections

In [ ]:
# Configure test folders for COCO object detection
test_folders = [
    ("Human ROI Test Dataset", os.path.join("src", "human_ROI_test")),
    ("Many Objects COCO Dataset", os.path.join("src", "many_objects_COCO")),
]

for dataset_name, rel_path in test_folders:
    folder_path = rel_path if os.path.exists(rel_path) else os.path.join("lab_works_2026_2027_1st_sem", "lab_1", rel_path)
    if not os.path.exists(folder_path):
        print(f"Directory not found: {folder_path}")
        continue

    img_files = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        img_files.extend(glob.glob(os.path.join(folder_path, ext)))
    img_files = sorted(img_files)

    print(f"==================== {dataset_name} ({len(img_files)} images) ====================")
    for img_path in img_files:
        img_name = os.path.basename(img_path)
        img_rgb = load_image(img_path)

        # Run YOLOv10 nano detection
        detected_img, detections = detect_yolov10_objects(img_rgb, conf_threshold=0.30)

        # Summarize detected classes and counts
        counts = {}
        for d in detections:
            counts[d['class_name']] = counts.get(d['class_name'], 0) + 1
        summary_str = ", ".join([f"{cls}: {cnt}" for cls, cnt in counts.items()]) if counts else "No detections"

        title = f"{img_name} | Total: {len(detections)} ({summary_str})"
        visualize_detections(img_rgb, detected_img, title=title)

## 3.3. MobileNetV3 (trained on COCO dataset)

* [PyTorch Torchvision Models Documentation](https://pytorch.org/vision/stable/models.html#object-detection)
* [MobileNetV3 Paper (Searching for MobileNetV3)](https://arxiv.org/abs/1905.02244)
* Architecture: SSDLite320 with MobileNetV3-Large backbone pre-trained on COCO dataset (`torchvision.models.detection.ssdlite320_mobilenet_v3_large`)

In [ ]:
import torch
import torchvision
from torchvision.models.detection import (
    ssdlite320_mobilenet_v3_large,
    SSDLite320_MobileNet_V3_Large_Weights
)
from torchvision.transforms import functional as F

# Select computation device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pre-trained MobileNetV3 Large SSDLite320 model with COCO weights
mobilenet_weights = SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
mobilenet_model = ssdlite320_mobilenet_v3_large(weights=mobilenet_weights)
mobilenet_model.to(device)
mobilenet_model.eval()

# Extract COCO categories from model metadata
MOBILENET_CLASSES = mobilenet_weights.meta["categories"]

print(f"MobileNetV3 (SSDLite320) loaded successfully on {device}!")
print(f"Total COCO categories supported: {len(MOBILENET_CLASSES)}")

In [ ]:
def detect_mobilenetv3_objects(
    image: np.ndarray,
    model: torch.nn.Module = mobilenet_model,
    categories: list = MOBILENET_CLASSES,
    conf_threshold: float = 0.30,
    dev: torch.device = device
) -> tuple[np.ndarray, list[dict]]:
    """
    Perform MobileNetV3-SSDLite object detection on an RGB image.
    Returns annotated RGB image and list of detection dictionaries.
    """
    h, w = image.shape[:2]

    # Convert RGB numpy array to PyTorch tensor [C, H, W] normalized to [0, 1]
    img_tensor = F.to_tensor(image).to(dev)

    with torch.no_grad():
        predictions = model([img_tensor])[0]

    boxes = predictions["boxes"].cpu().numpy()
    scores = predictions["scores"].cpu().numpy()
    labels = predictions["labels"].cpu().numpy()

    result_img = image.copy()
    detections = []

    for box, score, label_id in zip(boxes, scores, labels):
        if score >= conf_threshold:
            x1, y1, x2, y2 = box
            orig_x1 = max(0, min(w, int(round(x1))))
            orig_y1 = max(0, min(h, int(round(y1))))
            orig_x2 = max(0, min(w, int(round(x2))))
            orig_y2 = max(0, min(h, int(round(y2))))

            box_w = orig_x2 - orig_x1
            box_h = orig_y2 - orig_y1

            cls_name = categories[label_id] if 0 <= label_id < len(categories) else f"class_{label_id}"
            color = CLASS_COLORS.get(cls_name, (0, 200, 255))
            label = f"{cls_name} {score:.2f}"

            # Draw bounding box and label badge
            cv2.rectangle(result_img, (orig_x1, orig_y1), (orig_x2, orig_y2), color, 2)
            label_size, _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            badge_y1 = max(0, orig_y1 - label_size[1] - 6)
            badge_y2 = max(0, orig_y1)
            cv2.rectangle(
                result_img,
                (orig_x1, badge_y1),
                (orig_x1 + label_size[0] + 6, badge_y2),
                color,
                -1
            )
            cv2.putText(
                result_img,
                label,
                (orig_x1 + 3, max(label_size[1] + 1, orig_y1 - 3)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (255, 255, 255),
                1,
                cv2.LINE_AA
            )

            detections.append({
                "class_id": int(label_id),
                "class_name": cls_name,
                "score": float(score),
                "box": (orig_x1, orig_y1, box_w, box_h)
            })

    return result_img, detections

In [ ]:
# Configure test folders for MobileNetV3 COCO object detection
test_folders = [
    ("Human ROI Test Dataset", os.path.join("src", "human_ROI_test")),
    ("Many Objects COCO Dataset", os.path.join("src", "many_objects_COCO")),
]

for dataset_name, rel_path in test_folders:
    folder_path = rel_path if os.path.exists(rel_path) else os.path.join("lab_works_2026_2027_1st_sem", "lab_1", rel_path)
    if not os.path.exists(folder_path):
        print(f"Directory not found: {folder_path}")
        continue

    img_files = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        img_files.extend(glob.glob(os.path.join(folder_path, ext)))
    img_files = sorted(img_files)

    print(f"==================== {dataset_name} ({len(img_files)} images) ====================")
    for img_path in img_files:
        img_name = os.path.basename(img_path)
        img_rgb = load_image(img_path)

        # Run MobileNetV3 object detection
        detected_img, detections = detect_mobilenetv3_objects(img_rgb, conf_threshold=0.30)

        # Summarize detected classes and counts
        counts = {}
        for d in detections:
            counts[d['class_name']] = counts.get(d['class_name'], 0) + 1
        summary_str = ", ".join([f"{cls}: {cnt}" for cls, cnt in counts.items()]) if counts else "No detections"

        title = f"{img_name} | Total: {len(detections)} ({summary_str})"
        visualize_detections(img_rgb, detected_img, title=title)

# 4. Running segmentation models (coloring image pixels with it's corresponding class)

## 4.1. Torchvision DeepLabV3 (Semantic Segmentation)

* [PyTorch Torchvision Semantic Segmentation Documentation](https://pytorch.org/vision/stable/models.html#semantic-segmentation)
* [DeepLabV3 Paper (Rethinking Atrous Convolution for Semantic Image Segmentation)](https://arxiv.org/abs/1706.05587)
* Architecture: DeepLabV3 with ResNet-50 backbone pre-trained on COCO / PASCAL VOC (21 classes) using Atrous Spatial Pyramid Pooling (ASPP)

In [ ]:
from torchvision.models.segmentation import (
    deeplabv3_resnet50,
    DeepLabV3_ResNet50_Weights
)

# 21 PASCAL VOC class labels used by DeepLabV3
VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus',
    'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike',
    'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

# Generate unique deterministic colors for each segmentation class
np.random.seed(42)
VOC_COLORMAP = np.random.randint(0, 255, size=(len(VOC_CLASSES), 3), dtype=np.uint8)
VOC_COLORMAP[0] = [0, 0, 0]  # Background kept black

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pre-trained DeepLabV3 model with ResNet-50 backbone
deeplab_weights = DeepLabV3_ResNet50_Weights.DEFAULT
deeplab_transforms = deeplab_weights.transforms()
deeplab_model = deeplabv3_resnet50(weights=deeplab_weights).to(device)
deeplab_model.eval()

print(f"DeepLabV3 (ResNet-50) loaded successfully on {device}!")
print(f"Total semantic classes supported: {len(VOC_CLASSES)}")

In [ ]:
def overlay_semantic_mask(
    image: np.ndarray,
    mask: np.ndarray,
    colormap: np.ndarray = VOC_COLORMAP,
    alpha: float = 0.55
) -> np.ndarray:
    """
    Overlays a discrete integer class mask on an RGB image.
    Preserves original background where mask == 0.
    """
    colored_mask = colormap[mask % len(colormap)]
    blended = cv2.addWeighted(image, 1.0 - alpha, colored_mask, alpha, 0)
    bg_mask = (mask == 0)[:, :, None]
    return np.where(bg_mask, image, blended)


def segment_deeplabv3(
    image: np.ndarray,
    model: torch.nn.Module = deeplab_model,
    transforms = deeplab_transforms,
    dev: torch.device = device,
    alpha: float = 0.55
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """
    Perform semantic segmentation using DeepLabV3.
    Returns:
        blended_img: RGB image with class color overlays
        class_mask: 2D numpy array of predicted class indices
        detected_labels: List of detected non-background category names
    """
    h, w = image.shape[:2]
    pil_img = Image.fromarray(image)
    input_tensor = transforms(pil_img).unsqueeze(0).to(dev)

    with torch.no_grad():
        output = model(input_tensor)["out"][0]
        pred_mask = output.argmax(0).byte().cpu().numpy()

    class_mask = cv2.resize(pred_mask, (w, h), interpolation=cv2.INTER_NEAREST)
    blended_img = overlay_semantic_mask(image, class_mask, colormap=VOC_COLORMAP, alpha=alpha)

    unique_classes = np.unique(class_mask)
    detected_labels = [VOC_CLASSES[idx] for idx in unique_classes if idx != 0 and idx < len(VOC_CLASSES)]

    return blended_img, class_mask, detected_labels

In [ ]:
# Configure test folders for DeepLabV3 semantic segmentation
test_folders = [
    ("Human ROI Test Dataset", os.path.join("src", "human_ROI_test")),
    ("Many Objects COCO Dataset", os.path.join("src", "many_objects_COCO")),
]

for dataset_name, rel_path in test_folders:
    folder_path = rel_path if os.path.exists(rel_path) else os.path.join("lab_works_2026_2027_1st_sem", "lab_1", rel_path)
    if not os.path.exists(folder_path):
        print(f"Directory not found: {folder_path}")
        continue

    img_files = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        img_files.extend(glob.glob(os.path.join(folder_path, ext)))
    img_files = sorted(img_files)

    print(f"==================== {dataset_name} ({len(img_files)} images) ====================")
    for img_path in img_files:
        img_name = os.path.basename(img_path)
        img_rgb = load_image(img_path)

        # Run DeepLabV3 semantic segmentation
        blended_img, class_mask, labels = segment_deeplabv3(img_rgb, alpha=0.55)

        summary_str = ", ".join(labels) if labels else "No classes detected"
        title = f"{img_name} | DeepLabV3 Classes: ({summary_str})"
        visualize_detections(img_rgb, blended_img, title=title)

## 4.2. Torchvision Mask R-CNN (Instance Segmentation)

* [PyTorch Torchvision Instance Segmentation Documentation](https://pytorch.org/vision/stable/models.html#instance-segmentation)
* [Mask R-CNN Paper](https://arxiv.org/abs/1703.06870)
* Architecture: Mask R-CNN with ResNet-50 FPN V2 backbone pre-trained on COCO dataset (`torchvision.models.detection.maskrcnn_resnet50_fpn_v2`), generating bounding boxes, class labels, and pixel-level instance segmentation masks

In [ ]:
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn_v2,
    MaskRCNN_ResNet50_FPN_V2_Weights
)

# Load pre-trained Mask R-CNN model with ResNet-50 FPN V2 backbone
maskrcnn_weights = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
maskrcnn_model = maskrcnn_resnet50_fpn_v2(weights=maskrcnn_weights).to(device)
maskrcnn_model.eval()

# Extract COCO categories from metadata
MASKRCNN_CLASSES = maskrcnn_weights.meta["categories"]

print(f"Mask R-CNN (ResNet-50 FPN V2) loaded successfully on {device}!")
print(f"Total COCO categories supported: {len(MASKRCNN_CLASSES)}")

In [ ]:
def segment_maskrcnn_objects(
    image: np.ndarray,
    model: torch.nn.Module = maskrcnn_model,
    categories: list = MASKRCNN_CLASSES,
    conf_threshold: float = 0.50,
    mask_threshold: float = 0.50,
    dev: torch.device = device,
    alpha: float = 0.50
) -> tuple[np.ndarray, list[dict]]:
    """
    Perform instance segmentation using Mask R-CNN.
    Overlays colored instance masks, bounding boxes, and class labels.
    """
    h, w = image.shape[:2]
    img_tensor = F.to_tensor(image).to(dev)

    with torch.no_grad():
        predictions = model([img_tensor])[0]

    boxes = predictions["boxes"].cpu().numpy()
    scores = predictions["scores"].cpu().numpy()
    labels = predictions["labels"].cpu().numpy()
    masks = predictions["masks"].cpu().numpy()

    result_img = image.copy()
    instances = []

    np.random.seed(101)
    instance_colors = np.random.randint(50, 255, size=(len(boxes) + 20, 3), dtype=np.uint8)

    for i, (box, score, label_id, mask) in enumerate(zip(boxes, scores, labels, masks)):
        if score >= conf_threshold:
            color = tuple(map(int, instance_colors[i]))
            bin_mask = (mask[0] >= mask_threshold)

            # Apply colored instance mask overlay
            colored_layer = np.zeros_like(result_img, dtype=np.uint8)
            colored_layer[bin_mask] = color
            mask_indices = bin_mask[:, :, None]
            result_img = np.where(
                mask_indices,
                cv2.addWeighted(result_img, 1.0 - alpha, colored_layer, alpha, 0),
                result_img
            )

            # Draw bounding box and label badge
            x1, y1, x2, y2 = map(int, box)
            orig_x1 = max(0, min(w, x1))
            orig_y1 = max(0, min(h, y1))
            orig_x2 = max(0, min(w, x2))
            orig_y2 = max(0, min(h, y2))

            cls_name = categories[label_id] if 0 <= label_id < len(categories) else f"class_{label_id}"
            badge_text = f"{cls_name} {score:.2f}"

            cv2.rectangle(result_img, (orig_x1, orig_y1), (orig_x2, orig_y2), color, 2)
            label_size, _ = cv2.getTextSize(badge_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            badge_y1 = max(0, orig_y1 - label_size[1] - 6)
            badge_y2 = max(0, orig_y1)
            cv2.rectangle(
                result_img,
                (orig_x1, badge_y1),
                (orig_x1 + label_size[0] + 6, badge_y2),
                color,
                -1
            )
            cv2.putText(
                result_img,
                badge_text,
                (orig_x1 + 3, max(label_size[1] + 1, orig_y1 - 3)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (255, 255, 255),
                1,
                cv2.LINE_AA
            )

            instances.append({
                "instance_id": i,
                "class_id": int(label_id),
                "class_name": cls_name,
                "score": float(score),
                "box": (orig_x1, orig_y1, orig_x2 - orig_x1, orig_y2 - orig_y1)
            })

    return result_img, instances

In [ ]:
# Configure test folders for Mask R-CNN instance segmentation
test_folders = [
    ("Human ROI Test Dataset", os.path.join("src", "human_ROI_test")),
    ("Many Objects COCO Dataset", os.path.join("src", "many_objects_COCO")),
]

for dataset_name, rel_path in test_folders:
    folder_path = rel_path if os.path.exists(rel_path) else os.path.join("lab_works_2026_2027_1st_sem", "lab_1", rel_path)
    if not os.path.exists(folder_path):
        print(f"Directory not found: {folder_path}")
        continue

    img_files = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        img_files.extend(glob.glob(os.path.join(folder_path, ext)))
    img_files = sorted(img_files)

    print(f"==================== {dataset_name} ({len(img_files)} images) ====================")
    for img_path in img_files:
        img_name = os.path.basename(img_path)
        img_rgb = load_image(img_path)

        # Run Mask R-CNN instance segmentation
        annotated_img, instances = segment_maskrcnn_objects(img_rgb, conf_threshold=0.50, mask_threshold=0.50)

        # Summarize detected instance counts per category
        counts = {}
        for inst in instances:
            counts[inst['class_name']] = counts.get(inst['class_name'], 0) + 1
        summary_str = ", ".join([f"{cls}: {cnt}" for cls, cnt in counts.items()]) if counts else "No instances detected"

        title = f"{img_name} | Total Instances: {len(instances)} ({summary_str})"
        visualize_detections(img_rgb, annotated_img, title=title)

## 4.3. MediaPipe Image Segmenter (Edge / Real-Time Semantic Segmentation)

* [Google MediaPipe Image Segmenter Guide](https://ai.google.dev/edge/mediapipe/solutions/vision/image_segmenter)
* [MediaPipe DeepLabV3 TFLite Model Source](https://storage.googleapis.com/mediapipe-models/image_segmenter/deeplab_v3/float32/1/deeplab_v3.tflite)
* Architecture: MediaPipe Vision Tasks Image Segmenter with lightweight DeepLabV3 TFLite graph, optimized for on-device and real-time CPU/GPU semantic pixel segmentation

In [ ]:
import urllib.request
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Locate and prepare MediaPipe DeepLabV3 model file
MEDIAPIPE_MODEL_DIR = os.path.join("models")
if not os.path.exists(MEDIAPIPE_MODEL_DIR):
    MEDIAPIPE_MODEL_DIR = os.path.join("lab_works_2026_2027_1st_sem", "lab_1", "models")
os.makedirs(MEDIAPIPE_MODEL_DIR, exist_ok=True)

MEDIAPIPE_MODEL_PATH = os.path.join(MEDIAPIPE_MODEL_DIR, "deeplabv3.tflite")

# Download pre-trained MediaPipe DeepLabV3 model bundle if not present
if not os.path.exists(MEDIAPIPE_MODEL_PATH):
    url = "https://storage.googleapis.com/mediapipe-models/image_segmenter/deeplab_v3/float32/1/deeplab_v3.tflite"
    print(f"Downloading MediaPipe DeepLabV3 model to: {MEDIAPIPE_MODEL_PATH}...")
    try:
        urllib.request.urlretrieve(url, MEDIAPIPE_MODEL_PATH)
        print("Download complete.")
    except Exception as e:
        print(f"Download failed: {e}")


def create_mediapipe_segmenter(model_path: str = MEDIAPIPE_MODEL_PATH) -> vision.ImageSegmenter:
    """
    Initializes a MediaPipe Tasks Image Segmenter configured for categorical masks.
    """
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.ImageSegmenterOptions(
        base_options=base_options,
        output_category_mask=True,
        output_confidence_masks=False
    )
    return vision.ImageSegmenter.create_from_options(options)


mp_segmenter = None
if os.path.exists(MEDIAPIPE_MODEL_PATH):
    mp_segmenter = create_mediapipe_segmenter(MEDIAPIPE_MODEL_PATH)
    print(f"MediaPipe Image Segmenter loaded successfully from: {MEDIAPIPE_MODEL_PATH}")
else:
    print(f"MediaPipe model file not found at: {MEDIAPIPE_MODEL_PATH}")

In [ ]:
def segment_mediapipe(
    image: np.ndarray,
    segmenter: vision.ImageSegmenter,
    alpha: float = 0.55
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """
    Perform semantic segmentation using MediaPipe Image Segmenter.
    Returns:
        blended_img: RGB image with category color overlay
        category_mask: 2D numpy array of class indices
        detected_labels: List of detected non-background category names
    """
    # Convert numpy RGB image to MediaPipe Image format
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image)

    # Execute segmentation
    segmentation_result = segmenter.segment(mp_image)
    category_mask = segmentation_result.category_mask.numpy_view()

    # Blend category mask with original image
    blended_img = overlay_semantic_mask(image, category_mask, colormap=VOC_COLORMAP, alpha=alpha)

    unique_classes = np.unique(category_mask)
    detected_labels = [VOC_CLASSES[idx] for idx in unique_classes if idx != 0 and idx < len(VOC_CLASSES)]

    return blended_img, category_mask, detected_labels

In [ ]:
def segment_mediapipe(
    image: np.ndarray,
    segmenter: vision.ImageSegmenter,
    alpha: float = 0.55
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """
    Perform semantic segmentation using MediaPipe Image Segmenter.
    Ensures spatial consistency by resizing the output mask to match the input image.
    """
    h, w = image.shape[:2]

    # Convert numpy RGB image to MediaPipe Image format
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image)

    # Execute segmentation
    segmentation_result = segmenter.segment(mp_image)
    raw_category_mask = segmentation_result.category_mask.numpy_view()

    # Re-align mask to exact input dimensions to enable seamless pixel-wise blending
    category_mask = cv2.resize(raw_category_mask, (w, h), interpolation=cv2.INTER_NEAREST)

    # Blend category mask with original image
    blended_img = overlay_semantic_mask(image, category_mask, colormap=VOC_COLORMAP, alpha=alpha)

    unique_classes = np.unique(category_mask)
    detected_labels = [VOC_CLASSES[idx] for idx in unique_classes if idx != 0 and idx < len(VOC_CLASSES)]

    return blended_img, category_mask, detected_labels

In [ ]:
# Configure test folders for MediaPipe Image Segmenter
test_folders = [
    ("Human ROI Test Dataset", os.path.join("src", "human_ROI_test")),
    ("Many Objects COCO Dataset", os.path.join("src", "many_objects_COCO")),
]

if mp_segmenter is not None:
    for dataset_name, rel_path in test_folders:
        folder_path = rel_path if os.path.exists(rel_path) else os.path.join("lab_works_2026_2027_1st_sem", "lab_1", rel_path)
        if not os.path.exists(folder_path):
            print(f"Directory not found: {folder_path}")
            continue

        img_files = []
        for ext in ("*.jpg", "*.jpeg", "*.png"):
            img_files.extend(glob.glob(os.path.join(folder_path, ext)))
        img_files = sorted(img_files)

        print(f"==================== {dataset_name} ({len(img_files)} images) ====================")
        for img_path in img_files:
            img_name = os.path.basename(img_path)
            img_rgb = load_image(img_path)

            # Run MediaPipe semantic segmentation
            blended_img, category_mask, labels = segment_mediapipe(img_rgb, mp_segmenter, alpha=0.55)

            summary_str = ", ".join(labels) if labels else "No classes detected"
            title = f"{img_name} | MediaPipe Classes: ({summary_str})"
            visualize_detections(img_rgb, blended_img, title=title)
else:
    print("MediaPipe Segmenter is not available. Please ensure model file is downloaded.")

# 5. Running object tracking methods

# 6. Combining models 
(Movement identification with MOG2 -> Object detection -> Object tracking)